# Event Dummy

**4가지 데이터셋 생성 스크립트**
- EDA_1st_result.csv : 원본 1st data (레벨값)
- Derived_Variable.csv : 1st data가 1차 차분된 값 + 파생변수들 (oil_diff_target 포함)

**생성되는 데이터셋** (모두 타겟 변수는 oil_diff_target = OilPrice.diff().shift(-1)):
1) dataset1_raw_only        : 1st data 레벨값 + oil_diff_target (OilPrice 제외)
2) dataset2_raw_plus_derived: 1st data 레벨값 + 파생변수 + oil_diff_target (OilPrice 제외)
3) dataset3_diff_only       : 1st data 1차 차분 + oil_diff_target (차분된 OilPrice 제외)
4) dataset4_derived_full    : Derived_Variable 그대로 (차분된 1st + 파생변수 + target)

In [1]:
import pandas as pd
from pathlib import Path

# ===== 경로 설정 =====
INPUT_DIR = Path("../data/Finance_Final")
OUTPUT_DIR = Path("../data/Finance_Final")  # 사용자 지정 상대경로
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EDA_PATH = INPUT_DIR / "EDA_1st_result.csv"
DERIVED_PATH = INPUT_DIR / "Derived_Variable.csv"

TARGET_COL = "oil_diff_target"  # OilPrice 대신 사용할 타겟 변수


# ===== 데이터 로드 =====
df_raw = pd.read_csv(EDA_PATH, parse_dates=["date"])
df_derived = pd.read_csv(DERIVED_PATH, parse_dates=["date"])

print(f"[원본 1st data]      shape={df_raw.shape}, "
      f"기간={df_raw['date'].min().date()} ~ {df_raw['date'].max().date()}")
print(f"[Derived_Variable]   shape={df_derived.shape}, "
      f"기간={df_derived['date'].min().date()} ~ {df_derived['date'].max().date()}")


# ===== 컬럼 분류 =====
raw_cols = [c for c in df_raw.columns if c != "date"]

# 파생변수 컬럼 (Derived에만 있는 것)
derived_only_cols = [c for c in df_derived.columns if c not in df_raw.columns]

# Derived 안에서 1st data와 동일한 이름의 컬럼 (이미 1차 차분된 값)
diff_cols_in_derived = [c for c in df_raw.columns
                        if c in df_derived.columns and c != "date"]


def reorder_target_first(df: pd.DataFrame) -> pd.DataFrame:
    """date → oil_diff_target → 나머지 순으로 컬럼 정렬"""
    others = [c for c in df.columns if c not in ("date", TARGET_COL)]
    return df[["date", TARGET_COL] + others]


# ===== 1) 1st data만 (OilPrice → oil_diff_target 으로 교체) =====
dataset1 = df_raw.merge(
    df_derived[["date", TARGET_COL]],
    on="date",
    how="inner",
)
dataset1 = dataset1.drop(columns=["OilPrice"])
dataset1 = reorder_target_first(dataset1)


# ===== 2) 1st data + 파생변수 (OilPrice → oil_diff_target 으로 교체) =====
# 파생변수에 이미 oil_diff_target 포함되어 있음
dataset2 = df_raw.merge(
    df_derived[["date"] + derived_only_cols],
    on="date",
    how="inner",
)
dataset2 = dataset2.drop(columns=["OilPrice"])
dataset2 = reorder_target_first(dataset2)


# ===== 3) 1st data의 1차 차분만 (차분된 OilPrice → oil_diff_target 으로 교체) =====
# Derived에서 1st data와 동명 컬럼들(차분값) + oil_diff_target 만 사용
dataset3 = df_derived[["date"] + diff_cols_in_derived + [TARGET_COL]].copy()
dataset3 = dataset3.drop(columns=["OilPrice"])  # 차분된 OilPrice도 제거
dataset3 = reorder_target_first(dataset3)


# ===== 4) Derived_Variable 그대로 =====
dataset4 = df_derived.copy()
dataset4 = reorder_target_first(dataset4)


# ===== 저장 =====
out_files = {
    "dataset1_raw_only.csv":         dataset1,
    "dataset2_raw_plus_derived.csv": dataset2,
    "dataset3_diff_only.csv":        dataset3,
    "dataset4_derived_full.csv":     dataset4,
}

print(f"\n===== 저장 경로: {OUTPUT_DIR.resolve()} =====")
for fname, df in out_files.items():
    fpath = OUTPUT_DIR / fname
    df.to_csv(fpath, index=False)
    print(f"{fname:35s}  shape={df.shape}")


# ===== 요약 =====
print("\n===== 데이터셋 설명 (모두 타겟: oil_diff_target) =====")
print("1) dataset1_raw_only         : 1st data 레벨값 (OilPrice 제외) + oil_diff_target")
print("2) dataset2_raw_plus_derived : 1st data 레벨값 (OilPrice 제외) + 파생변수 (target 포함)")
print("3) dataset3_diff_only        : 1st data 1차 차분 (차분된 OilPrice 제외) + oil_diff_target")
print("4) dataset4_derived_full     : 1st data 1차 차분 + 파생변수 (target 포함)")

# 컬럼 확인용 출력
print(f"\n[dataset1 컬럼] {list(dataset1.columns)}")
print(f"[dataset2 컬럼] {list(dataset2.columns)}")
print(f"[dataset3 컬럼] {list(dataset3.columns)}")
print(f"[dataset4 컬럼] {list(dataset4.columns)}")

C:\Users\SAMSUNG\AppData\Local\Temp\ipykernel_14840\706245265.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


[원본 1st data]      shape=(4820, 14), 기간=2007-01-02 ~ 2026-03-16
[Derived_Variable]   shape=(4799, 28), 기간=2007-02-01 ~ 2026-03-16

===== 저장 경로: C:\Users\SAMSUNG\OneDrive\바탕 화면\비어플\Final_file\BAF-Finance2\data\Finance_Final =====
dataset1_raw_only.csv                shape=(4799, 14)
dataset2_raw_plus_derived.csv        shape=(4799, 28)
dataset3_diff_only.csv               shape=(4799, 13)
dataset4_derived_full.csv            shape=(4799, 28)

===== 데이터셋 설명 (모두 타겟: oil_diff_target) =====
1) dataset1_raw_only         : 1st data 레벨값 (OilPrice 제외) + oil_diff_target
2) dataset2_raw_plus_derived : 1st data 레벨값 (OilPrice 제외) + 파생변수 (target 포함)
3) dataset3_diff_only        : 1st data 1차 차분 (차분된 OilPrice 제외) + oil_diff_target
4) dataset4_derived_full     : 1st data 1차 차분 + 파생변수 (target 포함)

[dataset1 컬럼] ['date', 'oil_diff_target', 'RealInterestRate', 'CPI', 'DollarIndex', 'VIX', 'IndustryProduction', 'CPE', 'OilInventories', 'OPECProduction', 'OilProduction', 'TermSpread', 'TreasuryYield', 'Fed

## ** 공통 부분 - 하드코딩

In [ ]:
## 하드코딩
#방식 1. 단기충격 + 장기국면
#방식 2. 단기충격 중심
#방식 3. 이벤트 윈도우
import pandas as pd
import numpy as np

# --------------------------------------------
# 0. 데이터 준비
# --------------------------------------------
dataset_map = {
    'dataset1_raw_only': dataset1,
    'dataset2_raw_plus_derived': dataset2,
    'dataset3_diff_only': dataset3,
    'dataset4_derived_full': dataset4,
}

# date 컬럼을 DatetimeIndex로 변환
def prepare_dataset(df):
    df_out = df.copy()

    if not isinstance(df_out.index, pd.DatetimeIndex):
        if 'date' in df_out.columns:
            df_out['date'] = pd.to_datetime(df_out['date'])
            df_out = df_out.set_index('date').sort_index()
        else:
            raise ValueError("DatetimeIndex 또는 date 컬럼이 필요합니다.")

    return df_out

# 4가지 데이터셋 모두에 동일한 전처리 적용
prepared_datasets = {
    name: prepare_dataset(df)
    for name, df in dataset_map.items()
}

# 확인
for name, df in prepared_datasets.items():
    print(
        f"{name:30s} | shape={df.shape} | "
        f"period={df.index.min().date()} ~ {df.index.max().date()}"
    )

dataset1_raw_only              | shape=(4799, 13) | period=2007-02-01 ~ 2026-03-16
dataset2_raw_plus_derived      | shape=(4799, 27) | period=2007-02-01 ~ 2026-03-16
dataset3_diff_only             | shape=(4799, 12) | period=2007-02-01 ~ 2026-03-16
dataset4_derived_full          | shape=(4799, 27) | period=2007-02-01 ~ 2026-03-16


In [ ]:
# --------------------------------------------
# 1. 이벤트 기준일 / 기간 정의
# --------------------------------------------
# 기준일(anchor): 사건 발생 혹은 시장 충격이 본격화된 시점
# shock: 단기 충격 구간 (급격한 시장 반응)
# regime: 이후 지속되는 구조적 변화 구간,장기국면 (shock 이후 구간만 따로
#→ “충격(Shock) + 이후 체제 변화(Regime)를 나눠서 분석하겠다”는 설계

event_periods = {
    'gfc_2008': {
        'anchor': '2008-09-15',   # 리먼 사태 <기준일>
        'shock_start': '2008-09-15', #<단기 충격 구간>
        'shock_end':   '2008-12-31',
        'regime_start':'2009-01-01', #<장기 국면 구간>
        'regime_end':  '2009-06-30'
    },
    'opec_2014': { 
        'anchor': '2014-11-27',   # OPEC 감산 대신 점유율 유지 
        'shock_start': '2014-11-27',
        'shock_end':   '2015-01-31',
        'regime_start':'2015-02-01',
        'regime_end':  '2016-02-29'
    },
    'covid_2020': {
        'anchor': '2020-03-11',   # WHO 팬데믹 선언
        'shock_start': '2020-03-11',
        'shock_end':   '2020-05-31',
        'regime_start':'2020-06-01',
        'regime_end':  '2020-12-31'
    },
    'war_2022': {
        'anchor': '2022-02-24',   # 러시아-우크라이나 전쟁 발발
        'shock_start': '2022-02-24',
        'shock_end':   '2022-04-30',
        'regime_start':'2022-05-01',
        'regime_end':  '2022-12-31'
    }
}

In [9]:
# --------------------------------------------
# 2. 공통 함수
#본 분석에서는 이벤트 발생 이전 구간을 더미에 포함하지 않았다. 
#사건 발생 이전 기간까지 포함할 경우, 모델이 사건 발생을 사전에 알고 있었다는 해석상의 문제가 발생할 수 있기 때문 
#따라서 이벤트 윈도우는 사건 발생일을 기준으로 이후 20영업일로 설정하여, 사건 이후 시장에 반영되는 단기 충격을 포착하고자 하였다.
# --------------------------------------------
def add_period_dummy(df_in, col_name, start, end):
    df_in[col_name] = ((df_in.index >= pd.to_datetime(start)) &
                        (df_in.index <= pd.to_datetime(end))).astype(int)
    return df_in

def add_window_dummy(df_in, col_name, anchor, window_bdays=20):
    anchor = pd.to_datetime(anchor)
    start = anchor
    end   = anchor + pd.offsets.BDay(window_bdays)
    df_in[col_name] = ((df_in.index >= start) & (df_in.index <= end)).astype(int)
    return df_in
#df_in 은 anchor <= 날짜 <= anchor + 20영업일

In [10]:
# --------------------------------------------
# 3. 방식 1,2,3에 따른 변수 생성 함수
# --------------------------------------------

def make_hard_dummies(index):
    
    # 방식 1: 단기충격 + 장기국면
    df_hard_sl = pd.DataFrame(index=index)

    for evt, info in event_periods.items():
        # 단기충격
        df_hard_sl = add_period_dummy(
            df_hard_sl,
            f'{evt}_shock',
            info['shock_start'],
            info['shock_end']
        )
        
        # 장기국면 (shock 이후만 따로)
        df_hard_sl = add_period_dummy(
            df_hard_sl,
            f'{evt}_regime',
            info['regime_start'],
            info['regime_end']
        )

    hard_sl_cols = [c for c in df_hard_sl.columns if c.endswith('_shock') or c.endswith('_regime')]

    # 방식 2: 단기충격 중심
    df_hard_short = pd.DataFrame(index=index)

    for evt, info in event_periods.items():
        df_hard_short = add_period_dummy(
            df_hard_short,
            f'{evt}_short',
            info['shock_start'],
            info['shock_end']
        )

    hard_short_cols = [c for c in df_hard_short.columns if c.endswith('_short')]

    # 방식 3: 이벤트 윈도우 (+20 영업일)
    df_hard_window = pd.DataFrame(index=index)

    for evt, info in event_periods.items():
        df_hard_window = add_window_dummy(
            df_hard_window,
            f'{evt}_window',
            info['anchor'],
            window_bdays=20
        )

    hard_window_cols = [c for c in df_hard_window.columns if c.endswith('_window')]

    return {
        'shock_plus_regime': {
            'df': df_hard_sl,
            'cols': hard_sl_cols
        },
        'short_only': {
            'df': df_hard_short,
            'cols': hard_short_cols
        },
        'event_window': {
            'df': df_hard_window,
            'cols': hard_window_cols
        }
    }

In [11]:
# --------------------------------------------
# 4. 4개 dataset별 하드코딩 이벤트 더미 생성
# --------------------------------------------

hard_sets_by_dataset = {
    name: make_hard_dummies(df.index)
    for name, df in prepared_datasets.items()
}

# 확인
for dataset_name, hard_sets in hard_sets_by_dataset.items():
    print(f"\n==============================")
    print(f"{dataset_name}")
    print(f"==============================")

    # 이벤트 윈도우
    df_hard_window = hard_sets['event_window']['df']
    hard_window_cols = hard_sets['event_window']['cols']

    print("\n[이벤트 윈도우 변수]")
    print(hard_window_cols)
    print(df_hard_window[hard_window_cols].sum().sort_index())
    # hard_window_cols 는 anchor 날짜부터 +20 영업일(BDay)

    # 단기충격 + 장기국면
    df_hard_sl = hard_sets['shock_plus_regime']['df']
    hard_sl_cols = hard_sets['shock_plus_regime']['cols']

    print("\n[단기충격+장기국면 변수]")
    print(hard_sl_cols)
    print(df_hard_sl[hard_sl_cols].sum().sort_index())
    # hard_sl_cols 는 단기+장기

    # 단기충격 중심
    df_hard_short = hard_sets['short_only']['df']
    hard_short_cols = hard_sets['short_only']['cols']

    print("\n[단기충격 중심 변수]")
    print(hard_short_cols)
    print(df_hard_short[hard_short_cols].sum().sort_index())
    # hard_short_cols 는 단기



dataset1_raw_only

[이벤트 윈도우 변수]
['gfc_2008_window', 'opec_2014_window', 'covid_2020_window', 'war_2022_window']
covid_2020_window    21
gfc_2008_window      21
opec_2014_window     19
war_2022_window      21
dtype: int64

[단기충격+장기국면 변수]
['gfc_2008_shock', 'gfc_2008_regime', 'opec_2014_shock', 'opec_2014_regime', 'covid_2020_shock', 'covid_2020_regime', 'war_2022_shock', 'war_2022_regime']
covid_2020_regime    149
covid_2020_shock      56
gfc_2008_regime      124
gfc_2008_shock        76
opec_2014_regime     271
opec_2014_shock       43
war_2022_regime      169
war_2022_shock        46
dtype: int64

[단기충격 중심 변수]
['gfc_2008_short', 'opec_2014_short', 'covid_2020_short', 'war_2022_short']
covid_2020_short    56
gfc_2008_short      76
opec_2014_short     43
war_2022_short      46
dtype: int64

dataset2_raw_plus_derived

[이벤트 윈도우 변수]
['gfc_2008_window', 'opec_2014_window', 'covid_2020_window', 'war_2022_window']
covid_2020_window    21
gfc_2008_window      21
opec_2014_window     19
war_20

In [12]:
# dataset별 hard dummy 합계가 같은지 확인
check_rows = []

for dataset_name, hard_sets in hard_sets_by_dataset.items():
    for scheme_name, info in hard_sets.items():
        sums = info['df'][info['cols']].sum()

        for col, val in sums.items():
            check_rows.append({
                'dataset': dataset_name,
                'scheme': scheme_name,
                'dummy': col,
                'count': val
            })

hard_check = pd.DataFrame(check_rows)

display(
    hard_check.pivot_table(
        index=['scheme', 'dummy'],
        columns='dataset',
        values='count'
    )
)


dataset                              dataset1_raw_only  \
scheme            dummy                                  
event_window      covid_2020_window               21.0   
                  gfc_2008_window                 21.0   
                  opec_2014_window                19.0   
                  war_2022_window                 21.0   
shock_plus_regime covid_2020_regime              149.0   
                  covid_2020_shock                56.0   
                  gfc_2008_regime                124.0   
                  gfc_2008_shock                  76.0   
                  opec_2014_regime               271.0   
                  opec_2014_shock                 43.0   
                  war_2022_regime                169.0   
                  war_2022_shock                  46.0   
short_only        covid_2020_short                56.0   
                  gfc_2008_short                  76.0   
                  opec_2014_short                 43.0   
                  war_2022_short                  46.0   

dataset                              dataset2_raw_plus_derived  \
scheme            dummy                                          
event_window      covid_2020_window                       21.0   
                  gfc_2008_window                         21.0   
                  opec_2014_window                        19.0   
                  war_2022_window                         21.0   
shock_plus_regime covid_2020_regime                      149.0   
                  covid_2020_shock                        56.0   
                  gfc_2008_regime                        124.0   
                  gfc_2008_shock                          76.0   
                  opec_2014_regime                       271.0   
                  opec_2014_shock                         43.0   
                  war_2022_regime                        169.0   
                  war_2022_shock                          46.0   
short_only        covid_2020_short                        56.0   
                  gfc_2008_short                          76.0   
                  opec_2014_short                         43.0   
                  war_2022_short                          46.0   

dataset                              dataset3_diff_only  dataset4_derived_full  
scheme            dummy                                                         
event_window      covid_2020_window                21.0                   21.0  
                  gfc_2008_window                  21.0                   21.0  
                  opec_2014_window                 19.0                   19.0  
                  war_2022_window                  21.0                   21.0  
shock_plus_regime covid_2020_regime               149.0                  149.0  
                  covid_2020_shock                 56.0                   56.0  
                  gfc_2008_regime                 124.0                  124.0  
                  gfc_2008_shock                   76.0                   76.0  
                  opec_2014_regime                271.0                  271.0  
                  opec_2014_shock                  43.0                   43.0  
                  war_2022_regime                 169.0                  169.0  
                  war_2022_shock                   46.0                   46.0  
short_only        covid_2020_short                 56.0                   56.0  
                  gfc_2008_short                   76.0                   76.0  
                  opec_2014_short                  43.0                   43.0  
                  war_2022_short                   46.0                   46.0

In [7]:
# --------------------------------------------
# 이벤트별 하드코딩 더미 개수 요약
# --------------------------------------------

summary_rows = []

for dataset_name, hard_sets in hard_sets_by_dataset.items():
    # 각 방식별 df와 cols 꺼내기
    df_hard_sl = hard_sets['shock_plus_regime']['df']
    hard_sl_cols = hard_sets['shock_plus_regime']['cols']

    df_hard_short = hard_sets['short_only']['df']
    hard_short_cols = hard_sets['short_only']['cols']

    df_hard_window = hard_sets['event_window']['df']
    hard_window_cols = hard_sets['event_window']['cols']

    # 이벤트 이름 추출
    events = [
        c.replace('_shock', '')
        for c in hard_sl_cols
        if c.endswith('_shock')
    ]

    for evt in events:
        summary_rows.append({
            'dataset': dataset_name,
            'event': evt,
            'shock': df_hard_sl[f'{evt}_shock'].sum(),
            'regime': df_hard_sl[f'{evt}_regime'].sum(),
            'short_only': df_hard_short[f'{evt}_short'].sum(),
            'event_window': df_hard_window[f'{evt}_window'].sum(),
        })

hard_dummy_summary = pd.DataFrame(summary_rows)

display(hard_dummy_summary)

,dataset,event,shock,regime,short_only,event_window
0,dataset1_raw_only,gfc_2008,76,124,76,21
1,dataset1_raw_only,opec_2014,43,271,43,19
2,dataset1_raw_only,covid_2020,56,149,56,21
3,dataset1_raw_only,war_2022,46,169,46,21
4,dataset2_raw_plus_derived,gfc_2008,76,124,76,21
5,dataset2_raw_plus_derived,opec_2014,43,271,43,19
6,dataset2_raw_plus_derived,covid_2020,56,149,56,21
7,dataset2_raw_plus_derived,war_2022,46,169,46,21
8,dataset3_diff_only,gfc_2008,76,124,76,21
9,dataset3_diff_only,opec_2014,43,271,43,19


## ** 공통부분 - 평가함수

In [ ]:
# --------------------------------------------
# 1. 평가 함수
# --------------------------------------------
import numpy as np
import pandas as pd
import statsmodels.api as sm

from sklearn.pipeline import Pipeline
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV, LassoCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [17]:
# --------------------------------------------
# 1-1. OLS 기본 평가 함수
# --------------------------------------------
def fit_eval_ols(df_model, feature_cols, target_col='oil_diff_target', test_size=0.2):
    use_cols = feature_cols + [target_col]
    temp = df_model[use_cols].dropna().copy()

    split_idx = int(len(temp) * (1 - test_size))
    # train/test 분리
    train = temp.iloc[:split_idx].copy()
    test  = temp.iloc[split_idx:].copy()

    X_train = sm.add_constant(train[feature_cols], has_constant='add')
    y_train = train[target_col]

    X_test = sm.add_constant(test[feature_cols], has_constant='add')
    y_test = test[target_col]

    # OLS model fit
    model = sm.OLS(y_train, X_train).fit(cov_type='HC3')
    pred  = model.predict(X_test)

    result = {
        'algorithm': 'OLS',
        'n_train': len(train),
        'n_test': len(test),
        'MAE': mean_absolute_error(y_test, pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, pred)),
        'R2_test': r2_score(y_test, pred),
        'Adj_R2_train': model.rsquared_adj,
        'AIC_train': model.aic,
        'BIC_train': model.bic
    }

    return model, result
# OLS 회귀모형 학습 후 예측 성능 계산

In [18]:
# --------------------------------------------
# OLS 더미변수 계수 및 유의성 추출 함수
# --------------------------------------------
def extract_added_terms(model, added_cols, model_name, scheme_name):
    rows = []

    if len(added_cols) == 0:
        return pd.DataFrame()

    for col in added_cols:
        if col in model.params.index:
            rows.append({
                'scheme': scheme_name,
                'algorithm': 'OLS',
                'model': model_name,
                'variable': col,
                'coef': model.params[col],
                'p_value': model.pvalues[col],
                'significant_5%': model.pvalues[col] < 0.05
            })
    return pd.DataFrame(rows)
# 새로 추가한 더미 변수들의 계수와 p-value 추출

In [19]:
# --------------------------------------------
# 1-2. Ridge / Lasso 평가 함수
# --------------------------------------------
def fit_eval_regularized(df_model,feature_cols,target_col='oil_diff_target',algorithm='Ridge',
    test_size=0.2,random_state=42):

    use_cols = feature_cols + [target_col]
    temp = df_model[use_cols].dropna().copy()

    split_idx = int(len(temp) * (1 - test_size))
    # train/test 분리
    train = temp.iloc[:split_idx].copy()
    test = temp.iloc[split_idx:].copy()

    X_train = train[feature_cols]
    y_train = train[target_col]
    # Ridge/Lasso는 절편 내부처리하므로 sm.add_constant 사용할 필요 없음

    X_test = test[feature_cols]
    y_test = test[target_col]

    tscv = TimeSeriesSplit(n_splits=5)
    # 시계열 데이터 Split

    if algorithm == 'Ridge':
        model = Pipeline([
            ('scaler', StandardScaler()),
            ('model', RidgeCV(alphas=np.logspace(-4, 4, 50), cv=tscv))
            # 0.0001부터 10000까지 로그 간격으로 50개 alpha 후보
        ])

    elif algorithm == 'Lasso':
        model = Pipeline([
            ('scaler', StandardScaler()),
            ('model', LassoCV(
                alphas=np.logspace(-4, 2, 50),
                # 0.0001부터 100까지 로그 간격으로 50개 alpha 후보
                cv=tscv,
                max_iter=10000,
                random_state=random_state
            ))
        ])
    # Ridge/Lasso는 계수 크기에 벌점을 주는 모델이기 때문에 Scaling이 필요함. 

    else:
        raise ValueError("algorithm은 'Ridge' 또는 'Lasso'만 가능합니다.")

    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    best_alpha = model.named_steps['model'].alpha_

    result = {
        'algorithm': algorithm,
        'n_train': len(train),
        'n_test': len(test),
        'MAE': mean_absolute_error(y_test, pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, pred)),
        'R2_test': r2_score(y_test, pred),
        'Adj_R2_train': np.nan,
        'AIC_train': np.nan,
        'BIC_train': np.nan,
        # 이 모델에서는 Adj_R2_train, AIC_train, BIC_train을 계산하지 않음
        # OLS 기본 모델과 맞추기 위해 빈칸으로 둔다
        'best_alpha' : best_alpha
    }

    return model, result

In [20]:
# --------------------------------------------
# Ridge / Lasso 추가 더미 변수 계수 및 유의성 추출 함수
# --------------------------------------------
def extract_regularized_terms(model, added_cols, model_name, scheme_name, feature_cols, algorithm):
    rows = []

    if len(added_cols) == 0:
        return pd.DataFrame()

    fitted_model = model.named_steps['model']
    coef_map = dict(zip(feature_cols, fitted_model.coef_))

    for col in added_cols:
        if col in coef_map:
            rows.append({
                'scheme': scheme_name,
                'algorithm': algorithm,
                'model': model_name,
                'variable': col,
                'coef': coef_map[col],
                'p_value': np.nan,
                'significant_5%': np.nan
            })

    return pd.DataFrame(rows)

In [22]:
# --------------------------------------------
# 2. 하드코딩/조건기반/둘다 비교 함수
#    OLS + Ridge + Lasso
# --------------------------------------------
def compare_models_for_scheme(base_df, hard_df, cond_df, scheme_name,
                              target_col='oil_diff_target', base_features=None, algorithms=None):

    if base_features is None:
        raise ValueError("base_features를 넣어주세요.")
    
    if algorithms is None:
        algorithms = ['OLS', 'Ridge', 'Lasso']

    # 기존 원본 df에 없던 컬럼 = 새로 만든 더미 변수라고 간주
    hard_cols = [c for c in hard_df.columns if c not in base_df.columns]
    cond_cols = [c for c in cond_df.columns if c not in base_df.columns]

    # 병합
    df_model = base_df.copy()
    if hard_cols:
        df_model = df_model.join(hard_df[hard_cols], how='left')
    if cond_cols:
        df_model = df_model.join(cond_df[cond_cols], how='left')

    # 더미 변수 결측은 0 처리
    extra_cols = hard_cols + cond_cols
    if extra_cols:
        df_model[extra_cols] = df_model[extra_cols].fillna(0)

    model_specs = {
        'base_only': base_features,
        'hard_only': base_features + hard_cols,
        'cond_only': base_features + cond_cols,
        'hard_plus_cond': base_features + hard_cols + cond_cols
    }
    # base_only와 base에 이벤트 더미변수, 조건변수 추가한 모델 비교

    metric_rows = []
    coef_tables = []

    for algorithm in algorithms:
        for model_name, feature_cols in model_specs.items():

            if algorithm == 'OLS':
                model, metrics = fit_eval_ols(
                    df_model=df_model,
                    feature_cols=feature_cols,
                    target_col=target_col,
                    test_size=0.2
                )

            elif algorithm in ['Ridge', 'Lasso']:
                model, metrics = fit_eval_regularized(
                    df_model=df_model,
                    feature_cols=feature_cols,
                    target_col=target_col,
                    algorithm=algorithm,
                    test_size=0.2
                )

            else:
                raise ValueError(f"지원하지 않는 algorithm입니다: {algorithm}")

            metrics['scheme'] = scheme_name
            metrics['model'] = model_name
            metrics['n_features'] = len(feature_cols)

            metric_rows.append(metrics)

            # 추가 더미 변수 목록
            if model_name == 'base_only':
                added_cols = []
            elif model_name == 'hard_only':
                added_cols = hard_cols
            elif model_name == 'cond_only':
                added_cols = cond_cols
            else:
                added_cols = hard_cols + cond_cols

            # 추가 변수 계수 추출
            if algorithm == 'OLS':
                coef_tables.append(
                    extract_added_terms(
                        model=model,
                        added_cols=added_cols,
                        model_name=model_name,
                        scheme_name=scheme_name
                    )
                )

            elif algorithm in ['Ridge', 'Lasso']:
                coef_tables.append(
                    extract_regularized_terms(
                        model=model,
                        added_cols=added_cols,
                        model_name=model_name,
                        scheme_name=scheme_name,
                        feature_cols=feature_cols,
                        algorithm=algorithm
                    )
                )

    metric_df = pd.DataFrame(metric_rows)[[
        'scheme', 'algorithm', 'model', 'n_features', 'n_train', 'n_test',
        'MAE', 'RMSE', 'R2_test', 'Adj_R2_train', 'AIC_train', 'BIC_train','best_alpha'
    ]]

    coef_df = pd.concat(coef_tables, axis=0, ignore_index=True) if coef_tables else pd.DataFrame()

    return metric_df, coef_df, hard_cols, cond_cols

## 1. dataset1_raw_only

#### 1) 조건기반

In [13]:
# --------------------------------------------
# 1. 조건기반
# --------------------------------------------

import pandas as pd
import numpy as np

df_cond_1 = prepared_datasets['dataset1_raw_only'].copy()

if not isinstance(df_cond_1.index, pd.DatetimeIndex):
    if 'date' in df_cond_1.columns:
        df_cond_1['date'] = pd.to_datetime(df_cond_1['date'])
        df_cond_1 = df_cond_1.set_index('date').sort_index()
    else:
        raise ValueError("DatetimeIndex 또는 date 컬럼이 필요합니다.")


# 1) VIX > 30: 시장 불안
df_cond_1['cond_vix_gt_30'] = (df_cond_1['VIX'] > 30).astype(int)

# 2) TermSpread < 0: 장단기 금리 역전
df_cond_1['cond_termspread_inv'] = (df_cond_1['TermSpread'] < 0).astype(int)

# 3) OilInventories 큰 폭 감소
# OilInventories는 0 변화가 많으므로, 실제 변화가 발생한 날만 기준으로 하위 25% 분위수 계산
inventory_change_1 = df_cond_1['OilInventories'].diff()
inventory_nonzero_1 = inventory_change_1[inventory_change_1 != 0]
inventory_threshold_1 = inventory_nonzero_1.quantile(0.25)

df_cond_1['cond_inventory_draw'] = (inventory_change_1 < inventory_threshold_1).astype(int)

# 4) OPECProduction 큰 폭 감소
# OPECProduction도 0 변화가 많으므로, 실제 변화가 발생한 날만 기준으로 하위 25% 분위수 계산
opec_change_1 = df_cond_1['OPECProduction'].diff()
opec_nonzero_1 = opec_change_1[opec_change_1 != 0]
opec_threshold_1 = opec_nonzero_1.quantile(0.25)

df_cond_1['cond_opec_cut'] = (opec_change_1 < opec_threshold_1).astype(int)

cond_cols_1 = [
    'cond_vix_gt_30',
    'cond_termspread_inv',
    'cond_inventory_draw',
    'cond_opec_cut'
]

print("생성된 조건기반 변수")
print(cond_cols_1)

print("\n조건별 1의 개수")
print(df_cond_1[cond_cols_1].sum())

print("\n비율(%)")
print((df_cond_1[cond_cols_1].mean() * 100).round(2))

print("\n임계값 확인")
print("inventory_threshold_1:", inventory_threshold_1)
print("opec_threshold_1:", opec_threshold_1)
print("cond_opec_cut 1의 개수:", df_cond_1['cond_opec_cut'].sum())
print("cond_opec_cut 비율(%):", round(df_cond_1['cond_opec_cut'].mean() * 100, 2))


생성된 조건기반 변수
['cond_vix_gt_30', 'cond_termspread_inv', 'cond_inventory_draw', 'cond_opec_cut']

조건별 1의 개수
cond_vix_gt_30         446
cond_termspread_inv    595
cond_inventory_draw    250
cond_opec_cut           56
dtype: int64

비율(%)
cond_vix_gt_30          9.29
cond_termspread_inv    12.40
cond_inventory_draw     5.21
cond_opec_cut           1.17
dtype: float64

임계값 확인
inventory_threshold_1: -3115.75
opec_threshold_1: -179.12150537634443
cond_opec_cut 1의 개수: 56
cond_opec_cut 비율(%): 1.17


In [14]:
# --------------------------------------------
# 2. Inventory / OPEC 보정 전후 비교
# --------------------------------------------

def compare_nonzero_threshold(change, name):
    threshold_raw = change.quantile(0.25)
    cond_raw = (change < threshold_raw).astype(int)

    change_nonzero = change[change != 0]
    threshold_adj = change_nonzero.quantile(0.25)
    cond_adj = (change < threshold_adj).astype(int)

    print(f"\n[{name}]")
    print("보정 전 threshold:", threshold_raw)
    print("보정 전 count:", cond_raw.sum())
    print("보정 전 rate(%):", round(cond_raw.mean() * 100, 2))

    print("보정 후 threshold:", threshold_adj)
    print("보정 후 count:", cond_adj.sum())
    print("보정 후 rate(%):", round(cond_adj.mean() * 100, 2))


inventory_change_1 = df_cond_1['OilInventories'].diff()
opec_change_1 = df_cond_1['OPECProduction'].diff()

compare_nonzero_threshold(inventory_change_1, "OilInventories")
compare_nonzero_threshold(opec_change_1, "OPECProduction")


[OilInventories]
보정 전 threshold: 0.0
보정 전 count: 481
보정 전 rate(%): 10.02
보정 후 threshold: -3115.75
보정 후 count: 250
보정 후 rate(%): 5.21

[OPECProduction]
보정 전 threshold: 0.0
보정 전 count: 97
보정 전 rate(%): 2.02
보정 후 threshold: -179.12150537634443
보정 후 count: 56
보정 후 rate(%): 1.17


In [ ]:
# --------------------------------------------
# 3. 하드코딩 / 조건기반 / 하드코딩+조건기반 모델 비교 준비
# => dataset1_raw_only
# --------------------------------------------

import pandas as pd
import numpy as np
import statsmodels.api as sm

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 0) 기준 변수 설정
df_base_1 = prepared_datasets['dataset1_raw_only'].copy() # dataset1 기준 데이터

target_col_1 = 'oil_diff_target' # 타깃 변수

# 기본 설명변수 후보
# dataset1은 raw level 변수들 + oil_diff_target으로 구성되어 있으므로 target만 제외한 나머지를 base feature로 사용
base_features_1 = [
    c for c in df_base_1.columns
    if c != target_col_1
]

print("타깃:", target_col_1)

print("\n기본 설명변수:")
print(base_features_1)

print("\n누락된 기본 변수:")
missing_cols_1 = [
    c for c in base_features_1
    if c not in df_base_1.columns
]
print(missing_cols_1)


# 1) 이벤트 더미 변수 목록 확인
hard_sets_1 = hard_sets_by_dataset['dataset1_raw_only']

print("\n하드코딩 변수 세트:")
for name, info in hard_sets_1.items():
    print(name, ":", info['cols'])


# 2) 조건기반 변수 목록 확인
cond_features_1 = [
    c for c in cond_cols_1
    if c in df_cond_1.columns
]

print("\n조건기반 변수:")
print(cond_features_1)

print("\n누락된 조건기반 변수:")
missing_cond_cols_1 = [
    c for c in cond_cols_1
    if c not in df_cond_1.columns
]
print(missing_cond_cols_1)

타깃: oil_diff_target

기본 설명변수:
['RealInterestRate', 'CPI', 'DollarIndex', 'VIX', 'IndustryProduction', 'CPE', 'OilInventories', 'OPECProduction', 'OilProduction', 'TermSpread', 'TreasuryYield', 'FedFundsRate']

누락된 기본 변수:
[]

하드코딩 변수 세트:
shock_plus_regime : ['gfc_2008_shock', 'gfc_2008_regime', 'opec_2014_shock', 'opec_2014_regime', 'covid_2020_shock', 'covid_2020_regime', 'war_2022_shock', 'war_2022_regime']
short_only : ['gfc_2008_short', 'opec_2014_short', 'covid_2020_short', 'war_2022_short']
event_window : ['gfc_2008_window', 'opec_2014_window', 'covid_2020_window', 'war_2022_window']

조건기반 변수:
['cond_vix_gt_30', 'cond_termspread_inv', 'cond_inventory_draw', 'cond_opec_cut']

누락된 조건기반 변수:
[]


#### 2) 모델링

In [ ]:
# --------------------------------------------
# 3. 사용할 하드코딩 세트 자동 수집
#    dataset1_raw_only 기준
# --------------------------------------------

hard_sets_1 = hard_sets_by_dataset['dataset1_raw_only']

print("사용 가능한 하드코딩 세트:", list(hard_sets_1.keys()))

# --------------------------------------------
# 평가할 알고리즘 세트
# --------------------------------------------

algorithms = ['OLS', 'Ridge', 'Lasso']
print("사용할 평가 알고리즘:", algorithms)

In [ ]:
# --------------------------------------------
# 4. 모든 기간 방식에 대해 모델 비교 실행
#    dataset1_raw_only 기준
# --------------------------------------------

all_metric_dfs_1 = []
all_coef_dfs_1 = []

for scheme_name, hard_info in hard_sets_1.items():
    metric_df, coef_df, hard_cols_used, cond_cols_used = compare_models_for_scheme(
        base_df=df_base_1,
        hard_df=hard_info['df'],
        cond_df=df_cond_1[cond_cols_1],
        scheme_name=scheme_name,
        target_col=target_col_1,
        base_features=base_features_1,
        algorithms=algorithms
    )

    print(f"\n===== {scheme_name} =====")
    print("hard cols:", hard_cols_used)
    print("cond cols:", cond_cols_used)

    all_metric_dfs_1.append(metric_df)
    all_coef_dfs_1.append(coef_df)

result_1 = pd.concat(all_metric_dfs_1, ignore_index=True)
coef_result_1 = pd.concat(all_coef_dfs_1, ignore_index=True)

print(result_1['model'].value_counts())
display(result_1)
display(coef_result_1)

## 2. dataset2_raw_plus_derived

## 3. dataset3_diff_only

## 4. dataset4_derived_full